# 05 — Publish MVP outputs to HuggingFace Datasets

Uploads the full MVP run (`/Volumes/ds_work/alenj00/cap_cache/runs/mvp/`) to a **private** HuggingFace Dataset:

**Target:** `alenjani/cap-counterfactuals-dev` — practice / iteration. The canonical paper dataset (`alenjani/cap-counterfactuals`) is a separate repo we'll create when the paper run is ready.

**Estimated size:** ~600 MB (600 PNGs + parquet + CSVs + figures).
**Estimated upload time:** ~5–10 min on the cluster's network.

## 1. Install

In [ ]:
%pip install --upgrade huggingface_hub

In [ ]:
dbutils.library.restartPython()

## 2. Auth + dataset config

In [ ]:
from pathlib import Path
from huggingface_hub import HfApi, login

HF_WRITE_TOKEN = dbutils.secrets.get(scope="cap-secrets", key="hf-write-token")
login(token=HF_WRITE_TOKEN, write_permission=True)

REPO_ID = "alenjani/cap-counterfactuals-dev"
RUN_DIR = Path("/Volumes/ds_work/alenj00/cap_cache/runs/mvp")
PATH_IN_REPO = "runs/mvp"  # leaves room for runs/full/, runs/v2/, etc.

assert RUN_DIR.exists(), f"MVP run dir missing: {RUN_DIR}"
print(f"Source: {RUN_DIR}")
print(f"Target: {REPO_ID} → /{PATH_IN_REPO}/")

## 3. Create the dataset (idempotent)

In [ ]:
api = HfApi()
from huggingface_hub.errors import RepositoryNotFoundError

try:
    info = api.repo_info(repo_id=REPO_ID, repo_type="dataset")
    print(f"Dataset {REPO_ID} already exists (private={info.private})")
except RepositoryNotFoundError:
    api.create_repo(
        repo_id=REPO_ID,
        repo_type="dataset",
        private=True,
        exist_ok=True,
    )
    print(f"Created PRIVATE dataset: {REPO_ID}")

## 4. Write the dataset card (README.md on HF)

Uploaded as the dataset's landing-page README — what visitors see when they open the dataset on huggingface.co.

In [ ]:
card = '''---
license: mit
task_categories:
  - image-to-image
language:
  - en
tags:
  - fairness
  - counterfactual
  - face-generation
  - generative-ai
  - audit
pretty_name: Counterfactual Audit Pipeline (CAP) — Dev / Practice
size_categories:
  - 100<n<1K
---

# CAP — Counterfactual Audit Pipeline (Dev / Practice Dataset)

**Status:** Practice / dev dataset, NOT the canonical paper dataset.
**Canonical paper dataset (TBD):** `alenjani/cap-counterfactuals` (created at submission time).

## What this is

Synthetic counterfactual face images + audit predictions + statistical results,
produced by the [Counterfactual Audit Pipeline](https://github.com/alenjani/counterfactual-audit-pipeline)
for measuring intersectional fairness of facial-analysis systems.

## MVP run (50 ids × 12 variations = 600 images)

- **Generator:** Flux.1-dev + ControlNet-Union (pose) + PuLID + InsightFace antelopev2
- **Quantization:** NF4, resolution 1024², 28 inference steps
- **Auditor:** DeepFace (gender + age tasks)
- **Hardware:** 4× L4 GPUs on Databricks (Ray fan-out)

## Layout

```
runs/mvp/
├── generated/        600 PNGs + manifest.parquet
├── validation/       identity_scores.parquet (ArcFace cosine sim) + summary
├── audit/            predictions.parquet (DeepFace × gender + age × 600)
├── analysis/         flip rates, H1 ANOVA, H2 McNemars, H3 logit, intersectional table
└── viz/              dashboard.html + figures (PDF/SVG/PNG)
```

## Headline result

From `runs/mvp/analysis/h2_mcnemars.csv`:
DeepFace gender prediction flips significantly between Fitzpatrick I and Fitzpatrick VI on
PuLID-controlled identity counterfactuals — McNemar p=0.0, FDR-corrected p=0.0, **H2 supported**.

## Known issues (do NOT ship as-is for the paper)

1. Identity preservation = 0% at θ=0.5 (mean cosine sim ≈ -0.004). PuLID/IP-Adapter
   did not effectively inject identity in this MVP run. Must debug before paper.
2. H1 ANOVA + H3 ordinal logit empty: zero-variance issue (DeepFace returns "Man"/"Woman",
   our axis stores "male"/"female" — every row marks `error=1`, no variance for ANOVA).
3. Single auditor only. Paper needs 7-system comparison (AWS, Azure, Face++, Google,
   plus open-source DeepFace, InsightFace, ArcFace).

## License & ethics

MIT. Generated face images are synthetic counterfactuals from FairFace seeds (CC-BY 4.0).
Use is restricted to fairness / bias research; not for biometric identification or
training general-purpose face models.

## How to load

```python
from huggingface_hub import snapshot_download
snapshot_download(repo_id="alenjani/cap-counterfactuals-dev", repo_type="dataset", token="hf_...")
```
'''
Path("/local_disk0/cap_dataset_README.md").write_text(card)
print(f"Card length: {len(card)} chars")

## 5. Upload

`upload_folder` does a single multipart upload — uses HF's xet backend so 600 MB takes ~5-10 min.

In [ ]:
import time

t0 = time.time()
api.upload_file(
    path_or_fileobj="/local_disk0/cap_dataset_README.md",
    path_in_repo="README.md",
    repo_id=REPO_ID,
    repo_type="dataset",
    commit_message="Initialize dataset card",
)
print(f"Card uploaded in {time.time() - t0:.1f}s")

In [ ]:
t0 = time.time()
api.upload_folder(
    folder_path=str(RUN_DIR),
    path_in_repo=PATH_IN_REPO,
    repo_id=REPO_ID,
    repo_type="dataset",
    commit_message="MVP run: 600 counterfactuals + audit + analysis + viz",
    ignore_patterns=["*.tmp", "*~", "*incremental_manifest*"],
)
wall = time.time() - t0
print(f"\nUpload wall: {wall:.0f}s ({wall/60:.1f} min)")
print(f"Dataset URL: https://huggingface.co/datasets/{REPO_ID}")

## 6. Verify what landed

In [ ]:
files = api.list_repo_files(repo_id=REPO_ID, repo_type="dataset")
print(f"Files in {REPO_ID}: {len(files)}")
for f in sorted(files)[:25]:
    print(f"  {f}")
if len(files) > 25:
    print(f"  ... and {len(files) - 25} more")